# 01 — Exploratory Data Analysis & Embedding Generation
### AI Research Paper Intelligence System

This notebook covers the first half of the pipeline:

1. Load the raw ArXiv ML papers dataset from Hugging Face
2. Clean and prepare the text data
3. Generate sentence embeddings with `all-MiniLM-L6-v2`
4. Sanity-check embeddings using Cosine Similarity
5. Cache the results to disk for the search engine notebook

**Dataset:** [CShorten/ML-ArXiv-Papers](https://huggingface.co/datasets/CShorten/ML-ArXiv-Papers)

## 1. Load the raw dataset

We use the Hugging Face `datasets` library — the standard tool for downloading and processing NLP datasets.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers")

In [ ]:
import pandas as pd

df = dataset["train"].to_pandas()
df.columns

In [ ]:
df.head()

## 2. Clean the data

We only need the `title` and `abstract` columns — everything else (including
stray `Unnamed` index columns left over from the original CSV export) is
noise for our embedding model, which only cares about semantic content.

We also cap the dataset at 50,000 papers to keep embedding generation and
FAISS indexing fast enough to run comfortably on a laptop.

In [ ]:
df = df[['title', 'abstract']]
df = df.head(50000)
df.isnull().sum()

## 3. Build a combined text field

To generate one rich embedding per paper, we merge `title` and `abstract`
into a single `paper_text` field. Using the title alone loses depth; using
the abstract alone can miss the punchy, high-signal keywords in the title.
Combining both gives the sentence-transformer the fullest context.

In [ ]:
df["paper_text"] = df["title"] + " " + df["abstract"]
df[["paper_text"]].head()

In [ ]:
# Remove newline characters and stray whitespace so the transformer reads
# each paper as one smooth, continuous paragraph.
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()

print(df["paper_text"].iloc[0])

## 4. Load the embedding model

`all-MiniLM-L6-v2` is a compact sentence-transformer that maps text to a
384-dimensional vector, capturing semantic meaning rather than exact
keywords. It's a strong balance of speed and quality for this use case.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(type(model))

## 5. Quick sanity check on a single paper

Before committing to encoding all 50,000 rows, we test on one sample to
confirm the model and pipeline behave as expected.

In [ ]:
sample_text = df["paper_text"].iloc[0]

embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)   # (384,) -- one 384-dimensional vector

## 6. Sanity-check with Cosine Similarity

Before encoding the full dataset, we confirm that Cosine Similarity behaves
as expected: a paper compared with itself should score ~1.0 (identical
direction), while two different papers should score lower.

`cosine_similarity` requires 2D arrays (matrices), so we reshape each
1D embedding vector to shape `(1, -1)`.

In [ ]:
import sklearn
from sklearn.metrics.pairwise import cosine_similarity

sample_embeddings = model.encode(df["paper_text"].head(5).to_list())
print(sample_embeddings.shape)  # (5, 384) -- 5 papers, 384 dims each

# A paper compared with itself -> similarity should be ~1.0
self_similarity = cosine_similarity(
    sample_embeddings[0].reshape(1, -1),
    sample_embeddings[0].reshape(1, -1)
)
print("Self-similarity:", self_similarity[0][0])

# Compare paper 0 against papers 1-4
for i in range(1, 5):
    sim = cosine_similarity(
        sample_embeddings[0].reshape(1, -1),
        sample_embeddings[i].reshape(1, -1)
    )
    print(f"Similarity with Paper {i}: {sim[0][0]:.4f}")

## 7. Encode the full dataset

Now we run the model over all 50,000 papers. This is the most
compute-intensive step in the pipeline (can take 20-30 minutes on CPU), so
we cache the result to disk — if `arxiv_embeddings.npy` already exists, we
load it instantly instead of recomputing.

In [ ]:
import numpy as np
import os

index_path = "../data/arxiv_embeddings.npy"

if os.path.exists(index_path):
    print("Loading saved embeddings...")
    embeddings = np.load(index_path)
else:
    print("Generating embeddings for all papers...")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    os.makedirs("../data", exist_ok=True)
    np.save(index_path, embeddings)
    print("Embeddings saved successfully!")

print(embeddings.shape)

## 8. Save the cleaned dataframe

We persist the cleaned dataframe alongside the embeddings so the search
engine notebook can map similarity scores back to actual paper titles and
abstracts.

In [ ]:
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/cleaned_arxiv_papers.csv", index=False)
print("Cleaned DataFrame saved successfully!")

---
### Next steps
Continue to **`02_Search_Engine.ipynb`** to build the FAISS index and the
full search + summarization + keyword extraction pipeline.